# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [1]:
#Import packages we will need for this task

import re
import Bio.Seq
import Bio.SeqIO 

# Identify the ends of each gene and the constant sequence 

HA_end = "CCGGATTTGCATATAATGATGCACCAT"
NA_end = "CACGATAGATAAATAATAGTGCACCAT"
constant = "AGGCGGCCGC"

# Count a 'read' of HA/NA if the gene end is directly prior the constant sequence.
HA_pattern = re.compile(HA_end + constant)
NA_pattern = re.compile(NA_end + constant) 

In [2]:
# Here's where I add my function

def classify_read(seqread):
    """Classify a sequencing read as HA, NA, or neither based on the presence of gene ends and constant sequence."""
    # Flip the read to the forward orientation, as an upper-case string
    read_rc = str(Bio.Seq.Seq(str(seqread)).reverse_complement()).upper()

    # check the flipped read against the HA and NA patterns    
    if HA_pattern.search(read_rc):
        return "HA"
    if NA_pattern.search(read_rc):
        return "NA"
    return "neither"

In [3]:
# the helper given in lecture 9: it runs a parser function against a list of test cases
# and reports the results
def run_tests(parser_function, test_cases):
    """Run a parser against a list of test cases and report the results.

    Args:
        parser_function: The function to test. It takes a sequence and returns a barcode
            string, or neither.
        test_cases: A list of (sequence, expected result, label) tuples.

    Returns:
        The number of test cases that failed.
    """
    n_failed = 0

    for seq, expected, label in test_cases:
        result = parser_function(seq)
        if result == expected:
            print(f"PASS  {label}")
        else:
            n_failed += 1
            print(f"FAIL  {label}: {seq!r} gave {result!r}, expected {expected!r}")

    return f"{n_failed} tests failed"

In [4]:
test_cases = [
    # (read in file orientation, expected result, label)
    ("CTTCCTGGTCACGGTTGCGGCCGCCTATGGTGCATCATTATATGCAAATCCGG", "HA", "positive: real HA read"),
    ("GCTTAAGTTATTTAGTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTG", "NA", "positive: real NA read"),
    ("GCTTAAGTTATTTAGTGCGGCCGCCTAAAAAAAAAAAAAAAAAAAAAAAAAAA", "neither", "negative: gene end is neither HA nor NA"),
    ("gcttaagttatttagtgcggccgcctatggtgcactattatttatctatcgtg", "NA", "positive: lower-case NA read"),
]

run_tests(classify_read, test_cases)

PASS  positive: real HA read
PASS  positive: real NA read
PASS  negative: gene end is neither HA nor NA
PASS  positive: lower-case NA read


'0 tests failed'

In [5]:
# read every sequencing read from the FASTQ file
seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq"))

# start a counter for each possible outcome
num_HA = 0
num_NA = 0
num_neither = 0

# classify each read and add 1 to the matching counter
for read in seqreads:
    gene = classify_read(read.seq)   # .seq pulls the sequence out of the full record
    if gene == "HA":
        num_HA += 1
    elif gene == "NA":
        num_NA += 1
    else:
        num_neither += 1

print("Reads mapping to HA:", num_HA)
print("Reads mapping to NA:", num_NA)
print("Reads matching neither:", num_neither)
print("Total:", num_HA + num_NA + num_neither)

Reads mapping to HA: 5316
Reads mapping to NA: 3973
Reads matching neither: 711
Total: 10000


5,316 reads map to HA, while 3,973 map to NA. 711 reads matched to neither.

2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [6]:
# a valid barcode has exactly 16 bases after the constant sequence at the terminus of the flipped read.

barcode_pattern = re.compile(constant + r"(?P<barcode>[ACGTN]{16})$")

In [7]:
# Writing a function for the barcode

def read_barcode(seqread):
    """Pull the barcode out of a sequencing read.

    Args:
        seqread: A sequencing read (a Seq object or a string), in the
            orientation of the FASTQ file.

    Returns:
        The 16-nucleotide barcode as an upper-case string, or None if
        the read does not have a valid barcode.
    """
    # flip the read into the forward orientation, as an upper-case string for re
    read_rc = str(Bio.Seq.Seq(str(seqread)).reverse_complement()).upper()

    # look for a valid barcode after the constant sequence
    match = barcode_pattern.search(read_rc)
    if match is None:
        return None
    return match.group("barcode")

In [8]:
# Testing barcode reading function

barcode_test_cases = [
    # (read in file orientation, expected barcode, label)
    ("TTTTTTTTTTTTTTTTGCGGCCGCCT", "AAAAAAAAAAAAAAAA", "positive: clean read from lecture 9"),
    ("GCTTAAGTTATTTAGTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTG", "ACTAAATAACTTAAGC", "positive: real NA read"),
    ("TTTTTTTTTTTTTTTGCGGCCGCCT", None, "negative: barcode too short (15)"),
    ("TTTTTTTTTTTTTTTTTGCGGCCGCCT", None, "negative: barcode too long (17)"),
    ("TTTTTTTYTTTTTTTTGCGGCCGCCT", None, "negative: barcode has a non-nucleotide letter"),
]

run_tests(read_barcode, barcode_test_cases)

PASS  positive: clean read from lecture 9
PASS  positive: real NA read
PASS  negative: barcode too short (15)
PASS  negative: barcode too long (17)
PASS  negative: barcode has a non-nucleotide letter


'0 tests failed'

In [9]:
# count HA and NA reads with invalid barcodes

num_HA_invalid = 0
num_NA_invalid = 0

for read in seqreads:
    gene = classify_read(read.seq)       # which gene is this read from?
    barcode = read_barcode(read.seq)     # does it have a valid barcode?

    # count reads that belong to a gene but have no valid barcode
    if gene == "HA" and barcode is None:
        num_HA_invalid += 1
    elif gene == "NA" and barcode is None:
        num_NA_invalid += 1

print("HA reads without a valid barcode:", num_HA_invalid)
print("NA reads without a valid barcode:", num_NA_invalid)

HA reads without a valid barcode: 70
NA reads without a valid barcode: 63


HA has 70 reads without a valid barcode, while NA has 63 reads without a valid barcode. 

3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [10]:
# I'll begin by using the dictionaries tool to count each barcode 
# We'll use one dictionary for HA and one for NA

HA_barcode_counts = {}
NA_barcode_counts = {}

for read in seqreads:
    gene = classify_read(read.seq)     # which gene?
    barcode = read_barcode(read.seq)   # valid barcode, or None?

    # skip reads without a valid barcode (those were counted in Q2)
    if barcode is None:
        continue

    if gene == "HA":
        # first time seeing this barcode: start its count at 0
        if barcode not in HA_barcode_counts:
            HA_barcode_counts[barcode] = 0
        HA_barcode_counts[barcode] += 1
    elif gene == "NA":
        if barcode not in NA_barcode_counts:
            NA_barcode_counts[barcode] = 0
        NA_barcode_counts[barcode] += 1

print("Unique HA barcodes:", len(HA_barcode_counts))
print("Unique NA barcodes:", len(NA_barcode_counts))

Unique HA barcodes: 281
Unique NA barcodes: 138


In [11]:
#Now we want to find the most common barcode for HA

# keep track of the best barcode seen so far
top_HA = None
top_HA_count = 0

for barcode in HA_barcode_counts:
    if HA_barcode_counts[barcode] > top_HA_count:
        top_HA = barcode
        top_HA_count = HA_barcode_counts[barcode]

print("Most common HA barcode:", top_HA, "with", top_HA_count, "reads")

Most common HA barcode: CCCGACCCGACATTAA with 155 reads


In [12]:
# Now we want to find the most common barcode for NA

# keep track of the best barcode seen so far just as we did for HA

top_NA = None
top_NA_count = 0

for barcode in NA_barcode_counts:
    if NA_barcode_counts[barcode] > top_NA_count:
        top_NA = barcode
        top_NA_count = NA_barcode_counts[barcode]

print("Most common NA barcode:", top_NA, "with", top_NA_count, "reads")

Most common NA barcode: ACCAGTTCTCCCCGGG with 152 reads


The most common HA barcode is CCCGACCCGACATTAA with 155 reads.
The most common NA barcode is ACCAGTTCTCCCCGGG with 152 reads. 

Sources: Lecture 08 and 09 notebooks (patterns, run_tests helper, read_barcode structure); Claude (Anthropic) helped explain concepts and debug errors. 